# Data Validation — Sales Project

Apply agreed rules to the Bronze Sales, Customers, Products and Stores tables.

- Check required values, unique IDs, numeric values and dates.
- Validate relationships using master and reference tables.
- Report affected records with clear failure reasons.
- Summarize checked, passed, failed and unevaluated records.

Checks use business columns only. Records that do not meet a rule’s prerequisites are not evaluated for that rule.

Results are displayed and retained in notebook DataFrames for later review and cleaning. This notebook does not modify the Bronze tables or save the validation reports.

## Sales Validation

Check required fields, order ID uniqueness, quantities, amounts and dates. Confirm that sales IDs exist in their master tables and that totals match quantity multiplied by product price.

### Load Sales and Supporting Data

- Read Sales and exclude ingestion metadata.
- Count records and stop if the table is empty.
- Load customer IDs, product IDs and prices, and store IDs for relationship checks.

In [0]:
from pyspark.sql import functions as F
CATALOG = "sales_quality"
SCHEMA = "bronze_layer"

#Read only columns with sales data no ingestio column 
sales = spark.table(f"{CATALOG}.{SCHEMA}.sales")

business_columns = [
    column for column in sales.columns
    if not column.startswith("_")
]

sales = sales.select(*business_columns)
total_rows = sales.count()


if total_rows == 0 :
    raise ValueError("Sales is empty. Review ingestion ")

#load master fields for sales validation. sales row may have customer,product and store id but we need the master table to confirm this data is valed 
#without loading those columns you could only validate sales internally missing vlaues and duplicate orders but we can't detect the unknown id and so on 
customers = (
    spark.table(f"{CATALOG}.{SCHEMA}.customers")
    .select("customer_id")
)
products = (
    spark.table(f"{CATALOG}.{SCHEMA}.products")
    .select("product_id", "product_price")
)
stores = (
    spark.table(f"{CATALOG}.{SCHEMA}.stores")
    .select("store_id")
)
print(f"Sales records to validate : {total_rows:,}")

Sales records to validate : 2,004


### Prepare Validation Reporting

- Define a shared check for nulls and blank text.
- Prepare a helper to count checked, passed, affected and unevaluated rows.
- Retain affected business records with a failure reason and display examples.

In [0]:
# Store one summary and one issue DataFrame per rule.
validation_results = {}
sales_issues = {}
def is_missing(column):
    """Identify nulls and blank text."""
    return (
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    )

def report_check(rule, eligible, failed, severity="ERROR"):
    """Summarize a check and retain its affected business records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    # No eligible rows means the rule could not be evaluated.
    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    elif severity == "WARNING":
        status = "REVIEW"
    else:
        status = "FAIL"

    validation_results[rule] = (
        rule,
        severity,
        checked_count,
        checked_count - failed_count,
        failed_count,
        total_rows - checked_count,
        status,
    )

    # Keep original business values and explain the finding.
    sales_issues[rule] = (
        failed
        .select(*business_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    # Display a manageable sample; the DataFrame retains every failure.
    if failed_count > 0:
        display(sales_issues[rule].limit(20))


### Validate Required Fields

- Require all seven Sales business fields to be populated.
- Report nulls and blank text separately for each field.
- Record the result of each completeness rule.

In [0]:
#fields we need to validate 
#here we validate that all rows on those columns are required
required_columns = [
    "date",
    "order_id",
    "customer_id",
    "product_id",
    "units_sold",
    "total_amount",
    "store_id",
]
#check failed columns. values on this column must be exising to validate it 
for column in required_columns:
    failed= sales.filter(is_missing(column))

    report_check(
        rule =f"{column} must be existing",
        eligible = sales,
        failed = failed
    )

date must be existing: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
null,ORD001106,CS_Mor_46,P10073,5,170,St_Pho_ A,date must be existing
null,ORD001519,CS_Far_38,P10042,2,66,St_Alb_ N,date must be existing
null,ORD000904,CS_Jar_24,P10078,3,102,St_Los_Ca,date must be existing


order_id must be existing: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-11-04 00:00:00,null,CS_Jon_20,P10041,3,102,St_Bal_Ma,order_id must be existing
2020-01-13 00:00:00,null,CS_Del_6,P10009,6,432,St_Sea_Wa,order_id must be existing


customer_id must be existing: FAIL — 4 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-04-03 00:00:00,ORD001148,null,P10049,4,128,St_Pho_ A,customer_id must be existing
2021-08-12 00:00:00,ORD001773,null,P10023,2,66,St_Mia_Fl,customer_id must be existing
2020-02-21 00:00:00,ORD000684,null,P10064,4,128,St_Col_Oh,customer_id must be existing
2020-11-11 00:00:00,ORD000844,null,P10096,2,20,St_Cha_No,customer_id must be existing


product_id must be existing: PASS — 0 affected rows
units_sold must be existing: FAIL — 4 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-03-21 00:00:00,ORD001059,CS_Bry_48,P10020,null,99,St_Ind_In,units_sold must be existing
2021-08-07 00:00:00,ORD001503,CS_Hun_29,P10035,null,136,St_Alb_ N,units_sold must be existing
2020-07-24 00:00:00,ORD001625,CS_Bar_13,P10022,null,124,St_Ind_In,units_sold must be existing
2021-08-02 00:00:00,ORD000077,CS_Bea_11,P10035,null,136,St_Ind_In,units_sold must be existing


total_amount must be existing: PASS — 0 affected rows
store_id must be existing: PASS — 0 affected rows


### Validate Order ID Uniqueness

- Exclude missing order IDs already covered by completeness.
- Find populated IDs appearing more than once.
- Flag every record belonging to a repeated ID group.

This counts affected records, not just extra duplicate copies.

In [0]:
#validate order id is unique, on this case order id should be unique 
#missing orders validated on previous validation . so we will exclude them 
identified_sales = sales.filter(~is_missing("order_id"))

#find repated IDs 
repeated_ids = (
    identified_sales
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
    .select("order_id") #this shows only order_id column 
)
#display(repeated_ids)
#retrive all rows belonging to those repated IDs.
duplicated_orders = identified_sales.join(
          repeated_ids,
          on = "order_id",
          how = "inner"
)

report_check(
    rule ="order_id must be unique",
    eligible = identified_sales,
    failed= duplicated_orders
)



order_id must be unique: FAIL — 10 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A,order_id must be unique
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No,order_id must be unique
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh,order_id must be unique
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca,order_id must be unique
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N,order_id must be unique
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh,order_id must be unique
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No,order_id must be unique
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A,order_id must be unique
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca,order_id must be unique
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N,order_id must be unique


### Prepare Numeric and Date Checks

- Create temporary numeric quantities, numeric amounts and interpreted dates.
- Define valid quantities as finite positive whole numbers.
- Define valid amounts as finite positive numbers.
- Preserve original business values for reporting.

In [0]:
#all numeric values are saved as string . we need to interperate them first as numeric or date on temporary columns then validate them , then report the orginal values which still string as valid or not . we don't change anything on hte original dataframe 
sales_checked  = (
    sales
    .withColumn("quantity_number",
                F.expr("try_cast(units_sold AS DOUBLE)"),)
    .withColumn("amount_number",
                F.expr("try_cast(total_amount AS DOUBLE)"),)
    .withColumn("sale_date",
                F.expr("try_cast(`date` AS DATE)"),
)
)
#let's set teh rules of valid quantity and valid amount 
#valid quanitty . should be finit, positive and whole number 
#we create bolean column called valid quantity 
valid_quantity = F.coalesce(
    F.col("quantity_number").isNotNull()
    & ~F.isnan("quantity_number")
    & (F.abs(F.col("quantity_number")) != F.lit(float("inf")))
    & (F.col("quantity_number") > 0)
    & ((F.col("quantity_number") % 1) == 0),
    F.lit(False),
)

# Amounts must be finite positive numbers under our project rules.
valid_amount = F.coalesce(
    F.col("amount_number").isNotNull()
    & ~F.isnan("amount_number")
    & (F.abs(F.col("amount_number")) != F.lit(float("inf")))
    & (F.col("amount_number") > 0),
    F.lit(False),
)


### Validate Quantities and Amounts

- Check populated quantities against the positive-whole-number rule.
- Check populated amounts against the positive-number rule.
- Report conversion failures and values that violate these conditions.

Missing values remain covered by completeness checks.

In [0]:
#now validate both quantity and amount . 
#the approach still the same .first remove missing values .then you have eligible data to check . then we define the failed data and pass both to report_check function 
quantity_candidates = sales_checked.filter(~is_missing("units_sold"))
report_check(
    rule="units_sold must be a positive whole number",
    eligible=quantity_candidates,
    failed=quantity_candidates.filter(~valid_quantity),
)

amount_candidates = sales_checked.filter(~is_missing("total_amount"))
report_check(
    rule="total_amount must be a positive number",
    eligible=amount_candidates,
    failed=amount_candidates.filter(~valid_amount),
)

units_sold must be a positive whole number: FAIL — 6 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2022-06-21 00:00:00,ORD001349,CS_Com_17,P10021,five,170,St_Det_Mi,units_sold must be a positive whole number
2021-05-04 00:00:00,ORD000291,CS_Bry_48,P10007,0,126,St_Atl_Ge,units_sold must be a positive whole number
2020-08-15 00:00:00,ORD000304,CS_Mat_12,P10054,three,96,St_Ind_In,units_sold must be a positive whole number
2020-04-22 00:00:00,ORD000592,CS_But_25,P10022,-3,124,St_Atl_Ge,units_sold must be a positive whole number
2020-07-06 00:00:00,ORD000695,CS_Jar_24,P10100,-2,60,St_Atl_Ge,units_sold must be a positive whole number
2020-08-02 00:00:00,ORD000845,CS_Har_42,P10023,0,99,St_Bal_Ma,units_sold must be a positive whole number


total_amount must be a positive number: PASS — 0 affected rows


### Validate Dates

- Check whether populated date values represent valid calendar dates.
- Compare readable dates with today using the UTC session timezone.
- Report unreadable dates and future dates under separate rules.

Missing dates are covered by completeness checks.

In [0]:
# Set a consistent timezone for today's date.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# we exclude missing values here . we work on date values of the original data .
date_candidates = sales_checked.filter(
    ~is_missing("date")
)

report_check(
    rule="date must be a valid calendar date",
    eligible=date_candidates,
    failed=date_candidates.filter(F.col("sale_date").isNull()),
)

# now we still have unreadable dates that will give us Null on the sales_date. so we need to exclude them as we do usually. so now thes readable column is the ones that are eligible . and otherwise . but still didn't define failed column. 
#the failed column shouldn't be null on the main date and should be readable and shoudn't be after the current date 

# Only readable dates can be compared with today.
readable_dates = date_candidates.filter(
    F.col("sale_date").isNotNull()
)


report_check(
    rule="date must not be in the future",
    eligible=readable_dates,
    failed=readable_dates.filter(
        F.col("sale_date") > F.current_date()
    ),
)

date must be a valid calendar date: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-13-10,ORD001895,CS_Del_4,P10027,3,90,St_Pho_ A,date must be a valid calendar date
2020-04-31,ORD000664,CS_Col_39,P10073,2,68,St_Hou_Te,date must be a valid calendar date
2022-02-30,ORD000853,CS_Har_42,P10070,2,68,St_Alb_ N,date must be a valid calendar date


date must not be in the future: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2099-01-02 00:00:00,ORD000113,CS_Kel_44,P10046,2,64,St_Sea_Wa,date must not be in the future
2099-01-01 00:00:00,ORD000188,CS_Don_33,P10090,5,85,St_Ind_In,date must not be in the future
2099-01-03 00:00:00,ORD000992,CS_Ngu_31,P10002,6,444,St_Las_Ne,date must not be in the future


### Validate Sales IDs Against Master Tables

- Compare populated customer, product and store IDs with their master IDs.
- Find Sales records without a matching master record.
- Report unmatched IDs under separate rules.

In [0]:
#now we need to validate ids against the master data 
# Read Sales IDs >>> Read trusted master IDs >> Find Sales IDs with no match >> Report 

#pair each sales id with it's master table 
master_checks = [
    ("customer_id", customers),
    ("product_id", products),
    ("store_id", stores),
]

for column,master in master_checks:
    #first we choose rows where the values of itreated column is not null 
    candidates = sales.filter(~is_missing(column))
    master_ids = (
        master
        .filter(~is_missing(column)) # here we are making like validation for those columns on master tables 
        .select(column)
        .distinct()
    )

    unmatched_sales = candidates.join(
        master_ids,
        on=column,
        how="leftanti",
    )
    #it's time to report 
    report_check(
        rule=f"{column} must exist in its master table",
        eligible=candidates,
        failed=unmatched_sales,
    )


customer_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2022-02-12 00:00:00,ORD001211,CS_UNKNOWN,P10032,4,132,St_Alb_ N,customer_id must exist in its master table
2020-09-20 00:00:00,ORD001733,CS_UNKNOWN,P10054,2,64,St_Ind_In,customer_id must exist in its master table


product_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-12-31 00:00:00,ORD001300,CS_Sch_32,P99999,2,66,St_Det_Mi,product_id must exist in its master table
2021-06-17 00:00:00,ORD000537,CS_Stu_47,P99999,3,99,St_Hou_Te,product_id must exist in its master table


store_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-06-18 00:00:00,ORD000253,CS_Dav_37,P10033,4,140,ST_UNKNOWN,store_id must exist in its master table
2020-03-27 00:00:00,ORD000554,CS_Har_42,P10016,3,96,ST_UNKNOWN,store_id must exist in its master table


### Validate Sales Totals

- Prepare usable product prices and stop if populated product IDs are repeated.
- Check sales with valid quantities, valid amounts and matching usable prices.
- Calculate the expected amount as quantity multiplied by product price.
- Flag differences of 0.005 or more and display the calculation.

The project assumes no taxes, discounts or returns. Records with unusable inputs are not evaluated for this rule.

In [0]:
#Validate the calculated sales amount
# the approach is get sales product_id then find it's product price from products table then converts units sold to number and product_price to number as well and multiply them then compare with total_amount which needs to be converted to number as well 

#we convert product_price on products table to numeric
product_prices = products.withColumn(
    "reference_price",
    F.expr("try_cast(product_price AS DOUBLE)",)
)
#let's check if there is any products have more than one id 
# we work on original vaues  
ambiguous_prices = (
    product_prices
    .filter(~is_missing("product_id"))
    .groupBy("product_id")
    .count()
    .filter(F.col("count")>1)
)

if ambiguous_prices.count()>0 : 
    raise ValueError(
        "products contain repated IDs.")


#we need to keep only finite positive values for the producted converted prices 
product_prices = (
    product_prices
    .filter(
        F.col("reference_price").isNotNull()
        & ~F.isnan("reference_price")
        & (F.abs(F.col("reference_price")) != F.lit(float("inf"))) #reject positive and negative infinity 
        & (F.col("reference_price") > 0) # keep only positive values. Zero is rejected as well 
    )
    .select("product_id", "reference_price") # we dodn't need the main price column any more 
)


# Evaluate totals only when the necessary inputs are usable.
amount_comparison = (
    sales_checked
    .filter(valid_quantity & valid_amount) # this one rely on that we have defined column names on their origin clumn expression 
    .join(product_prices, on="product_id", how="inner")  
    .withColumn(
        "expected_amount",
        F.col("quantity_number") * F.col("reference_price"),
    )
)

incorrect_totals = amount_comparison.filter(
    F.abs(
        F.col("amount_number") - F.col("expected_amount")
    ) >= 0.005
)

report_check(
    rule="total_amount must equal quantity multiplied by product price",
    eligible=amount_comparison,
    failed=incorrect_totals,
)

# Show the calculation behind each mismatch.
display(
    incorrect_totals.select(
        "order_id",
        "product_id",
        "units_sold",
        "reference_price",
        "total_amount",
        "expected_amount",
    )
)


total_amount must equal quantity multiplied by product price: FAIL — 5 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-03-20 00:00:00,ORD000052,CS_Leo_14,P10037,4,153,St_Ind_In,total_amount must equal quantity multiplied by product price
2021-10-07 00:00:00,ORD000170,CS_Tur_45,P10034,2,85,St_Las_Ne,total_amount must equal quantity multiplied by product price
2021-06-07 00:00:00,ORD000290,CS_Smi_10,P10095,5,75,St_Det_Mi,total_amount must equal quantity multiplied by product price
2022-01-03 00:00:00,ORD000329,CS_Mar_40,P10094,4,73,St_Los_Ca,total_amount must equal quantity multiplied by product price
2021-08-28 00:00:00,ORD000583,CS_Dav_37,P10010,5,415,St_Los_Ca,total_amount must equal quantity multiplied by product price


order_id,product_id,units_sold,reference_price,total_amount,expected_amount
ORD000052,P10037,4,32.0,153,128.0
ORD000170,P10034,2,30.0,85,60.0
ORD000290,P10095,5,10.0,75,50.0
ORD000329,P10094,4,12.0,73,48.0
ORD000583,P10010,5,78.0,415,390.0


### Sales Validation Summary

- Display counts and status for each Sales rule.
- Show rules with the most affected records first.
- Keep unevaluated records separate from passed records.

One record can fail multiple rules, so affected counts must not be added together to count unique invalid sales.

In [0]:
#display all validation summary 
# Build the final summary from all recorded rule results.
sales_validation_summary = spark.createDataFrame(
    list(validation_results.values()),
    """
    rule string,
    severity string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    sales_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,severity,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
order_id must be unique,ERROR,2002,1992,10,2,FAIL
units_sold must be a positive whole number,ERROR,2000,1994,6,4,FAIL
total_amount must equal quantity multiplied by product price,ERROR,1992,1987,5,12,FAIL
customer_id must be existing,ERROR,2004,2000,4,0,FAIL
units_sold must be existing,ERROR,2004,2000,4,0,FAIL
date must be a valid calendar date,ERROR,2001,1998,3,3,FAIL
date must be existing,ERROR,2004,2001,3,0,FAIL
date must not be in the future,ERROR,1998,1995,3,6,FAIL
customer_id must exist in its master table,ERROR,2000,1998,2,4,FAIL
order_id must be existing,ERROR,2004,2002,2,0,FAIL


## Customers Validation

Check that required customer fields are populated and customer IDs are unique.

Gender and city receive completeness checks only. Customer home cities are not restricted to store locations.

### Load Customers and Prepare Results

- Read all customer business columns.
- Count records and stop if the table is empty.
- Initialize containers for customer summaries and affected records.

In [0]:
customers = spark.table(f"{CATALOG}.{SCHEMA}.customers")
# Select business columns only.
customer_columns = [
    column for column in customers.columns
    if not column.startswith("_")
]

customers = customers.select(*customer_columns)

customer_total_rows = customers.count()

if customer_total_rows == 0:
    raise ValueError("Customers is empty. Review ingestion first.")

# Store customer rule summaries and affected records.
customer_validation_results = []
customer_issues = {}

print(f"Customer records to validate: {customer_total_rows}")

Customer records to validate: 49


### Validate Required Fields

- Require customer ID, first name, last name, gender and city.
- Flag nulls and blank text in each required field.
- Record counts and retain failed records with their reasons.

In [0]:
#we would start by definig the required columns . We will use the same columns as in the original table
# Required fields under this project's customer rules.
required_customer_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "gender",
    "city",
]

for column in required_customer_columns:

    rule = f"{column} must be populated"

    # Find records with a missing required value.
    failed = customers.filter(is_missing(column))
    failed_count = failed.count()

    # Every customer record is eligible for completeness checks.
    passed_count = customer_total_rows - failed_count
    status = "PASS" if failed_count == 0 else "FAIL"

    customer_validation_results.append((
        rule,
        customer_total_rows,
        passed_count,
        failed_count,
        0,  # No records are excluded from this check.
        status,
    ))

    # Retain every failed record with its reason.
    customer_issues[rule] = failed.withColumn(
        "failure_reason",
        F.lit(rule),
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    # Show examples only when failures exist.
    if failed_count > 0:
        display(customer_issues[rule].limit(20))

customer_id must be populated: PASS — 0 affected rows
first_name must be populated: PASS — 0 affected rows
last_name must be populated: PASS — 0 affected rows
gender must be populated: PASS — 0 affected rows
city must be populated: PASS — 0 affected rows


### Validate Customer ID Uniqueness

- Exclude missing customer IDs from this check.
- Find populated IDs appearing more than once.
- Flag all records belonging to those repeated IDs.
- Record checked, passed, affected and unevaluated counts.

In [0]:
# now we will do same thing with customer id uniqueness 

# exclude null values 
identified_customers = customers.filter(
    ~is_missing("customer_id")
)

# Find customer IDs appearing more than once.
repeated_customer_ids = (
    identified_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

# Retrieve all records belonging to repeated IDs.
duplicate_customers = (
    identified_customers
    .join(
        repeated_customer_ids,
        on="customer_id",
        how="inner",
    )
    .select(*customer_columns)
)

checked_count = identified_customers.count()
failed_count = duplicate_customers.count()
passed_count = checked_count - failed_count

# Missing IDs are not evaluated for uniqueness.
not_evaluated_count = customer_total_rows - checked_count

if checked_count == 0:
    status = "NOT_EVALUATED"
elif failed_count == 0:
    status = "PASS"
else:
    status = "FAIL"

rule = "customer_id must be unique"

customer_validation_results.append((
    rule,
    checked_count,
    passed_count,
    failed_count,
    not_evaluated_count,
    status,
))

# Retain duplicate records for inspection.
customer_issues[rule] = duplicate_customers.withColumn(
    "failure_reason",
    F.lit(rule),
)

print(f"{rule}: {status} — {failed_count} affected rows")

if failed_count > 0:
    display(
        customer_issues[rule]
        .orderBy("customer_id")
        .limit(20)
    )

customer_id must be unique: PASS — 0 affected rows


### Customers Validation Summary

Display the results of the five required-field rules and the customer ID uniqueness rule. Use affected counts and statuses to identify checks requiring attention.

In [0]:
# Create one summary row per customer validation rule.
customer_validation_summary = spark.createDataFrame(
    customer_validation_results,
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    customer_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
city must be populated,49,49,0,0,PASS
customer_id must be populated,49,49,0,0,PASS
customer_id must be unique,49,49,0,0,PASS
first_name must be populated,49,49,0,0,PASS
gender must be populated,49,49,0,0,PASS
last_name must be populated,49,49,0,0,PASS


## Products Validation

Check required fields, product ID uniqueness and positive numeric prices. Validate categories against the approved reference and show available alias corrections without applying them.

### Load Products and Category References

- Read all product business columns and count records.
- Load approved categories for validation.
- Load category aliases to help explain possible corrections.

In [0]:
# Read the full Products table from Bronze.
products = spark.table(f"{CATALOG}.{SCHEMA}.products")

# Exclude ingestion metadata.
product_columns = [
    column for column in products.columns
    if not column.startswith("_")
]

products = products.select(*product_columns)
product_total_rows = products.count()

if product_total_rows == 0:
    raise ValueError("Products is empty. Review ingestion first.")

#We will use refernces tables for the validation of products
# Load the approved category values.
approved_categories = (
    spark.table(f"{CATALOG}.{SCHEMA}.product_categories")
    .select("category_name")
)

# Load known spelling variations and their approved corrections.
category_aliases = (
    spark.table(f"{CATALOG}.{SCHEMA}.category_aliases")
    .select("raw_value", "correct_value")
)

print(f"Product records to validate: {product_total_rows}")

Product records to validate: 100


### Prepare Product Reporting

- Initialize product result containers.
- Define a helper to summarize each rule.
- Retain affected business records with failure reasons and display examples.

In [0]:
#we build same helper functions 
# Separate containers for Products validation.
product_validation_results = {}
product_issues = {}

def report_product_check(rule, eligible, failed):
    """Record a product rule's results and affected records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    else:
        status = "FAIL"

    # Store one summary per rule.
    product_validation_results[rule] = (
        rule,
        checked_count,
        checked_count - failed_count,
        failed_count,
        product_total_rows - checked_count,
        status,
    )

    # Keep original business values and add the failure reason.
    product_issues[rule] = (
        failed
        .select(*product_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    if failed_count > 0:
        display(product_issues[rule].limit(20))

In [0]:
#we would start by validating required fields 
required_product_columns = [
    "product_id",
    "product_name",
    "product_price",
    "product_category",
]

for column in required_product_columns:

    # Find missing values in the current required field.
    failed = products.filter(is_missing(column))

    report_product_check(
        rule=f"{column} must be populated",
        eligible=products,
        failed=failed,
    )

product_id must be populated: PASS — 0 affected rows
product_name must be populated: PASS — 0 affected rows
product_price must be populated: PASS — 0 affected rows
product_category must be populated: PASS — 0 affected rows


### Validate Required Fields

Require product ID, product name, product price and product category. Flag nulls and blank text, then record the result for each field.

### Validate Product ID Uniqueness

- Exclude missing product IDs already covered by completeness.
- Find populated IDs appearing more than once.
- Flag all records sharing those IDs to identify ambiguous product records.

In [0]:
#Validate product ID uniqueness
# Missing IDs are covered by the completeness check.
identified_products = products.filter(
    ~is_missing("product_id")
)
# Find populated IDs used by multiple records.
repeated_product_ids = (
    identified_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .select("product_id")
)

# Retrieve every record belonging to a repeated ID.
duplicate_products = identified_products.join(
    repeated_product_ids,
    on="product_id",
    how="inner",
)

report_product_check(
    rule="product_id must be unique",
    eligible=identified_products,
    failed=duplicate_products,
)

product_id must be unique: PASS — 0 affected rows


### Validate Product Prices

- Temporarily convert product prices to numbers.
- Check populated prices for conversion failures, NaN, infinity, zero and negative values.
- Require finite positive prices and report failures.

Missing prices are handled by completeness checks.

In [0]:
#Validate product prices
#convert first product_price into number 
product_price_data = products.withColumn(
    "parsed_price",
    F.expr("try_cast(product_price AS DOUBLE)"),
)
#now we need to exlude values failed to be converted 
price_candidates = product_price_data.filter(
    ~is_missing("product_price")
) #now those are eligible values . but we need to validate them 

# Reject conversion failures, NaN, infinity, zero and negatives.
#first set the rule 
valid_price = F.coalesce(
    F.col("parsed_price").isNotNull()
    & ~F.isnan("parsed_price")
    & (F.abs(F.col("parsed_price")) != F.lit(float("inf")))
    & (F.col("parsed_price") > 0),
    F.lit(False),
)
#second apply it to get failed ones 
invalid_prices = price_candidates.filter(~valid_price)

report_product_check(
    rule="product_price must be a finite positive number",
    eligible=price_candidates,
    failed=invalid_prices,
)

product_price must be a finite positive number: PASS — 0 affected rows


### Validate Approved Categories

- Compare populated product categories exactly as stored with the approved list.
- Identify products whose category has no approved match.
- Report those records as category-validation failures.

In [0]:
#Validate categories against the approved reference
# Prepare the approved category lookup.
category_lookup = (
    approved_categories
    .filter(~is_missing("category_name"))
    .select(
        F.col("category_name").alias("product_category")
    )
    .distinct()
)

# Missing categories are already covered by completeness.
category_candidates = products.filter(
    ~is_missing("product_category")
)

# Find categories absent from the approved list.
invalid_product_categories = category_candidates.join(
    category_lookup,
    on="product_category",
    how="left_anti",
)

report_product_check(
    rule="product_category must match an approved category",
    eligible=category_candidates,
    failed=invalid_product_categories,
)

product_category must match an approved category: FAIL — 3 affected rows


product_id,product_name,product_price,product_category,failure_reason
P10001,Air purifier,66,electronics,product_category must match an approved category
P10014,oven gloves,35,Unknown Category,product_category must match an approved category
P10043,Ankle Training Socks,31,Clothng,product_category must match an approved category


### Review Category Corrections

- Match failed categories with known aliases.
- Display suggested approved categories where a mapping exists.
- Keep unmapped categories visible for investigation.

This is a review report, not an additional validation rule. No corrections are applied.

In [0]:
#Show whether failed categories have known corrections
#rename names of reference table 
alias_lookup = category_aliases.select(
    F.col("raw_value").alias("product_category"),
    F.col("correct_value").alias("suggested_category"),
)


# Attach known corrections to failed product categories.
product_category_review = (
    invalid_product_categories
    .join(
        alias_lookup,
        on="product_category",
        how="left",
    )
    .withColumn(
        "review_note",
        F.when(
            F.col("suggested_category").isNotNull(),
            "Known alias: correction available",
        ).otherwise(
            "No approved alias: review required"
        ),
    )
    .select(
        "product_id",
        "product_name",
        "product_category",
        "suggested_category",
        "review_note",
    )
)

display(
    product_category_review.orderBy("product_id")
)

product_id,product_name,product_category,suggested_category,review_note
P10001,Air purifier,electronics,Electronics,Known alias: correction available
P10014,oven gloves,Unknown Category,null,No approved alias: review required
P10043,Ankle Training Socks,Clothng,Clothing,Known alias: correction available


### Products Validation Summary

Display the results of required-field, product ID, price and category checks. Show affected counts and rule statuses to guide later review.

In [0]:
#display validation summary for the whole products table 
product_validation_summary = spark.createDataFrame(
    list(product_validation_results.values()),
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    product_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
product_category must match an approved category,100,97,3,0,FAIL
product_category must be populated,100,100,0,0,PASS
product_id must be populated,100,100,0,0,PASS
product_id must be unique,100,100,0,0,PASS
product_name must be populated,100,100,0,0,PASS
product_price must be a finite positive number,100,100,0,0,PASS
product_price must be populated,100,100,0,0,PASS


## Stores Validation

Check required fields and store ID uniqueness. Compare each city, state and location combination with the approved reference, then display mismatches for review.

### Load Stores and Location References

- Read store business columns and count records.
- Stop if Stores is empty.
- Load approved city, state and location combinations for validation.

In [0]:
#Load Stores and the location reference
# Read the Stores table from Bronze.
stores = spark.table(f"{CATALOG}.{SCHEMA}.stores")

# Exclude ingestion metadata.
store_columns = [
    column for column in stores.columns
    if not column.startswith("_")
]

stores = stores.select(*store_columns)
store_total_rows = stores.count()

if store_total_rows == 0:
    raise ValueError("Stores is empty. Review ingestion first.")

# Load approved location combinations.
store_locations = (
    spark.table(f"{CATALOG}.{SCHEMA}.store_locations")
    .select("city", "state", "location")
)

print(f"Store records to validate: {store_total_rows}")

Store records to validate: 14


### Prepare Store Reporting

- Initialize store result containers.
- Define a helper to summarize each rule.
- Retain affected business records with failure reasons and display examples.

In [0]:
#Prepare store validation reporting. build same helper functions 
# Store summaries and affected records separately for Stores.
store_validation_results = {}
store_issues = {}

def report_store_check(rule, eligible, failed):
    """Record a store rule's results and affected records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    else:
        status = "FAIL"

    # Keep one summary per rule.
    store_validation_results[rule] = (
        rule,
        checked_count,
        checked_count - failed_count,
        failed_count,
        store_total_rows - checked_count,
        status,
    )

    # Keep original business values with a failure reason.
    store_issues[rule] = (
        failed
        .select(*store_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    if failed_count > 0:
        display(store_issues[rule].limit(20))

### Validate Required Fields

Require store ID, city, state and location. Flag nulls and blank text, then record the completeness result for each field.

In [0]:
#start by validation required fields 
required_store_columns = [
    "store_id",
    "city",
    "state",
    "location",
]

for column in required_store_columns:

    # Find records missing the current required value.
    failed = stores.filter(is_missing(column))

    report_store_check(
        rule=f"{column} must be populated",
        eligible=stores,
        failed=failed,
    )

store_id must be populated: PASS — 0 affected rows
city must be populated: PASS — 0 affected rows
state must be populated: PASS — 0 affected rows
location must be populated: PASS — 0 affected rows


### Validate Store ID Uniqueness

- Exclude missing store IDs already covered by completeness.
- Find populated IDs appearing more than once.
- Flag every record belonging to a repeated ID group.

In [0]:
#Validate store ID uniqueness
# Keep records eligible for the uniqueness check.
identified_stores = stores.filter(
    ~is_missing("store_id")
)

# Find IDs appearing more than once.
repeated_store_ids = (
    identified_stores
    .groupBy("store_id")
    .count()
    .filter(F.col("count") > 1)
    .select("store_id")
)

# Retrieve every record belonging to a repeated ID.
duplicate_stores = identified_stores.join(
    repeated_store_ids,
    on="store_id",
    how="inner",
)

report_store_check(
    rule="store_id must be unique",
    eligible=identified_stores,
    failed=duplicate_stores,
)

store_id must be unique: PASS — 0 affected rows


### Validate Location Combinations

- Check stores with populated city, state and location fields.
- Compare all three values together with the approved reference.
- Report combinations without an exact match.

Individually valid location values can still form an incorrect combination.

In [0]:
#Validate the complete location combination
#we need to check if the store three values are valid together City+State+Location
#Read all soters >> Exclude rows with missing city,state or location >> compare all three colummns with store_locations refrence table >>Report those stores as Failures

# Keep records with all three location fields populated.
location_candidates = stores.filter(
    ~is_missing("city")
    & ~is_missing("state")
    & ~is_missing("location")
)


# Prepare complete, distinct approved combinations.
location_lookup = (
    store_locations
    .filter(
        ~is_missing("city")
        & ~is_missing("state")
        & ~is_missing("location")
    )
    .distinct()
)

# Find store combinations without an exact reference match.
invalid_store_locations = location_candidates.join(
    location_lookup,
    on=["city", "state", "location"], #join on all three columns 
    how="left_anti",
)

report_store_check(
    rule="City, state and location must match an approved combination",
    eligible=location_candidates,
    failed=invalid_store_locations,
)

City, state and location must match an approved combination: FAIL — 2 affected rows


store_id,city,state,location,failure_reason
St_Atl_Ge,Atlanta,Florida,East Coast,"City, state and location must match an approved combination"
St_Mia_Fl,Miami,Florida,Midwest,"City, state and location must match an approved combination"


### Review Location Mismatches

- Display failed stores alongside reference values matched by city.
- Keep stores whose city is absent from the reference visible.
- Use the comparison to investigate the mismatch.

Reference values are shown for review; no replacements are applied.

In [0]:
# Show reference values beside the failures
# Rename reference fields so we can compare them with stored values.
expected_locations = location_lookup.select(
    "city",
    F.col("state").alias("reference_state"),
    F.col("location").alias("reference_location"),
)

# Keep all failures, including cities absent from the reference.
store_location_review = (
    invalid_store_locations
    .join(
        expected_locations,
        on="city",
        how="left",
    )
    .withColumn(
        "review_note",
        F.when(
            F.col("reference_state").isNull(),
            "City absent from reference: review required",
        ).otherwise(
            "Compare stored values with the approved combination"
        ),
    )
    .select(
        "store_id",
        "city",
        "state",
        "reference_state",
        "location",
        "reference_location",
        "review_note",
    )
)

display(
    store_location_review.orderBy("city", "store_id")
)

store_id,city,state,reference_state,location,reference_location,review_note
St_Atl_Ge,Atlanta,Florida,Georgia,East Coast,East Coast,Compare stored values with the approved combination
St_Mia_Fl,Miami,Florida,Florida,Midwest,East Coast,Compare stored values with the approved combination


### Stores Validation Summary

Display the results of required-field, store ID and location-combination checks. Show affected counts and statuses for later review.

In [0]:
#Display the Stores validation summary
store_validation_summary = spark.createDataFrame(
    list(store_validation_results.values()),
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    store_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
"City, state and location must match an approved combination",14,12,2,0,FAIL
city must be populated,14,14,0,0,PASS
location must be populated,14,14,0,0,PASS
state must be populated,14,14,0,0,PASS
store_id must be populated,14,14,0,0,PASS
store_id must be unique,14,14,0,0,PASS


### Combine Validation Summaries

- Add the source table name to each summary.
- Use the same columns across all four tables.
- Preserve Sales severity values and label the other checks as errors.


In [0]:
# Add table names and align the severity column.
summary_reports = [
    sales_validation_summary
        .withColumn("table_name", F.lit("sales")),

    customer_validation_summary
        .withColumn("table_name", F.lit("customers"))
        .withColumn("severity", F.lit("ERROR")),

    product_validation_summary
        .withColumn("table_name", F.lit("products"))
        .withColumn("severity", F.lit("ERROR")),

    store_validation_summary
        .withColumn("table_name", F.lit("stores"))
        .withColumn("severity", F.lit("ERROR")),
]

# Define a consistent column order.
summary_columns = [
    "table_name",
    "rule",
    "severity",
    "checked_rows",
    "passed_rows",
    "affected_rows",
    "not_evaluated_rows",
    "status",
]

# Start with Sales, then add the other summaries.
combined_validation_summary = summary_reports[0].select(
    *summary_columns
)

for report in summary_reports[1:]:
    combined_validation_summary = (
        combined_validation_summary
        .unionByName(report.select(*summary_columns))
    )

display(
    combined_validation_summary.orderBy(
        "table_name",
        F.desc("affected_rows"),
        "rule",
    )
)

table_name,rule,severity,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
customers,city must be populated,ERROR,49,49,0,0,PASS
customers,customer_id must be populated,ERROR,49,49,0,0,PASS
customers,customer_id must be unique,ERROR,49,49,0,0,PASS
customers,first_name must be populated,ERROR,49,49,0,0,PASS
customers,gender must be populated,ERROR,49,49,0,0,PASS
customers,last_name must be populated,ERROR,49,49,0,0,PASS
products,product_category must match an approved category,ERROR,100,97,3,0,FAIL
products,product_category must be populated,ERROR,100,100,0,0,PASS
products,product_id must be populated,ERROR,100,100,0,0,PASS
products,product_id must be unique,ERROR,100,100,0,0,PASS


### Combine Validation Issues

- Collect failed records from each table’s issues dictionary.
- Include the table name, record ID, severity and failure reason.
- Store the original business fields together as JSON text because the tables have different columns.

One record can appear multiple times when it fails multiple rules. Missing IDs remain null, and duplicate records remain visible.

In [0]:
# Each entry identifies a table, its key and its issue reports.
issue_sources = [
    ("sales", "order_id", sales_issues),
    ("customers", "customer_id", customer_issues),
    ("products", "product_id", product_issues),
    ("stores", "store_id", store_issues),
]

standardized_issues = []

for table_name, id_column, issue_dictionary in issue_sources:

    for rule, issue_df in issue_dictionary.items():

        # Preserve each table's original business fields.
        record_columns = [
            column for column in issue_df.columns
            if column != "failure_reason"
            and not column.startswith("_")
        ]

        # Sales supports warnings; the other current rules are errors.
        severity = (
            validation_results[rule][1]
            if table_name == "sales"
            else "ERROR"
        )

        # Convert different table structures into one report structure.
        report = issue_df.select(
            F.lit(table_name).alias("table_name"),
            F.col(id_column).cast("string").alias("record_id"),
            F.lit(severity).alias("severity"),
            F.col("failure_reason"),
            F.to_json(
                F.struct(
                    *[F.col(column) for column in record_columns]
                ),
                options={"ignoreNullFields": "false"},
            ).alias("record_values"),
        )

        standardized_issues.append(report)

# Combine all rule-level reports, including empty ones.
combined_validation_issues = standardized_issues[0]

for report in standardized_issues[1:]:
    combined_validation_issues = (
        combined_validation_issues.unionByName(report)
    )

display(
    combined_validation_issues.orderBy(
        "table_name",
        "record_id",
        "failure_reason",
    )
)

table_name,record_id,severity,failure_reason,record_values
products,P10001,ERROR,product_category must match an approved category,"{""product_id"":""P10001"",""product_name"":""Air purifier"",""product_price"":""66"",""product_category"":""electronics""}"
products,P10014,ERROR,product_category must match an approved category,"{""product_id"":""P10014"",""product_name"":""oven gloves"",""product_price"":""35"",""product_category"":""Unknown Category""}"
products,P10043,ERROR,product_category must match an approved category,"{""product_id"":""P10043"",""product_name"":""Ankle Training Socks"",""product_price"":""31"",""product_category"":""Clothng""}"
sales,null,ERROR,order_id must be existing,"{""date"":""2020-01-13 00:00:00"",""order_id"":null,""customer_id"":""CS_Del_6"",""product_id"":""P10009"",""units_sold"":""6"",""total_amount"":""432"",""store_id"":""St_Sea_Wa""}"
sales,null,ERROR,order_id must be existing,"{""date"":""2021-11-04 00:00:00"",""order_id"":null,""customer_id"":""CS_Jon_20"",""product_id"":""P10041"",""units_sold"":""3"",""total_amount"":""102"",""store_id"":""St_Bal_Ma""}"
sales,ORD000052,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2020-03-20 00:00:00"",""order_id"":""ORD000052"",""customer_id"":""CS_Leo_14"",""product_id"":""P10037"",""units_sold"":""4"",""total_amount"":""153"",""store_id"":""St_Ind_In""}"
sales,ORD000077,ERROR,units_sold must be existing,"{""date"":""2021-08-02 00:00:00"",""order_id"":""ORD000077"",""customer_id"":""CS_Bea_11"",""product_id"":""P10035"",""units_sold"":null,""total_amount"":""136"",""store_id"":""St_Ind_In""}"
sales,ORD000113,ERROR,date must not be in the future,"{""date"":""2099-01-02 00:00:00"",""order_id"":""ORD000113"",""customer_id"":""CS_Kel_44"",""product_id"":""P10046"",""units_sold"":""2"",""total_amount"":""64"",""store_id"":""St_Sea_Wa""}"
sales,ORD000170,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2021-10-07 00:00:00"",""order_id"":""ORD000170"",""customer_id"":""CS_Tur_45"",""product_id"":""P10034"",""units_sold"":""2"",""total_amount"":""85"",""store_id"":""St_Las_Ne""}"
sales,ORD000188,ERROR,date must not be in the future,"{""date"":""2099-01-01 00:00:00"",""order_id"":""ORD000188"",""customer_id"":""CS_Don_33"",""product_id"":""P10090"",""units_sold"":""5"",""total_amount"":""85"",""store_id"":""St_Ind_In""}"


### Save the Validation Reports

- Create a schema for quality reports.
- Give both reports the same snapshot ID and recording time.
- Save them as Delta tables, replacing the previous report snapshots.

The recording time identifies when these reports were saved. Bronze business tables remain unchanged.

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

QUALITY_SCHEMA = "quality_layer"

# Identify this pair of report snapshots.
report_snapshot_id = str(uuid4())
report_recorded_at = datetime.now(timezone.utc)

# Create the destination schema if needed.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{QUALITY_SCHEMA}`"
)

# Add the same snapshot information to both reports.
summary_to_save = (
    combined_validation_summary
    .withColumn("report_snapshot_id", F.lit(report_snapshot_id))
    .withColumn("report_recorded_at", F.lit(report_recorded_at))
)

issues_to_save = (
    combined_validation_issues
    .withColumn("report_snapshot_id", F.lit(report_snapshot_id))
    .withColumn("report_recorded_at", F.lit(report_recorded_at))
)

# Destination tables contain the latest validation reports.
SUMMARY_TABLE = f"{CATALOG}.{QUALITY_SCHEMA}.validation_summary"
ISSUES_TABLE = f"{CATALOG}.{QUALITY_SCHEMA}.validation_issues"

# Replace the previous summary report.
(
    summary_to_save.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SUMMARY_TABLE)
)

# Replace the previous issues report.
(
    issues_to_save.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ISSUES_TABLE)
)

print(f"Saved: {SUMMARY_TABLE}")
print(f"Saved: {ISSUES_TABLE}")

Saved: sales_quality.quality_layer.validation_summary
Saved: sales_quality.quality_layer.validation_issues


In [0]:
saved_issues = spark.table(ISSUES_TABLE)
display(saved_issues)

table_name,record_id,severity,failure_reason,record_values,report_snapshot_id,report_recorded_at
sales,ORD000052,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2020-03-20 00:00:00"",""order_id"":""ORD000052"",""customer_id"":""CS_Leo_14"",""product_id"":""P10037"",""units_sold"":""4"",""total_amount"":""153"",""store_id"":""St_Ind_In""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000170,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2021-10-07 00:00:00"",""order_id"":""ORD000170"",""customer_id"":""CS_Tur_45"",""product_id"":""P10034"",""units_sold"":""2"",""total_amount"":""85"",""store_id"":""St_Las_Ne""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000290,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2021-06-07 00:00:00"",""order_id"":""ORD000290"",""customer_id"":""CS_Smi_10"",""product_id"":""P10095"",""units_sold"":""5"",""total_amount"":""75"",""store_id"":""St_Det_Mi""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000329,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2022-01-03 00:00:00"",""order_id"":""ORD000329"",""customer_id"":""CS_Mar_40"",""product_id"":""P10094"",""units_sold"":""4"",""total_amount"":""73"",""store_id"":""St_Los_Ca""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000583,ERROR,total_amount must equal quantity multiplied by product price,"{""date"":""2021-08-28 00:00:00"",""order_id"":""ORD000583"",""customer_id"":""CS_Dav_37"",""product_id"":""P10010"",""units_sold"":""5"",""total_amount"":""415"",""store_id"":""St_Los_Ca""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000291,ERROR,units_sold must be a positive whole number,"{""date"":""2021-05-04 00:00:00"",""order_id"":""ORD000291"",""customer_id"":""CS_Bry_48"",""product_id"":""P10007"",""units_sold"":""0"",""total_amount"":""126"",""store_id"":""St_Atl_Ge""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000304,ERROR,units_sold must be a positive whole number,"{""date"":""2020-08-15 00:00:00"",""order_id"":""ORD000304"",""customer_id"":""CS_Mat_12"",""product_id"":""P10054"",""units_sold"":""three"",""total_amount"":""96"",""store_id"":""St_Ind_In""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000592,ERROR,units_sold must be a positive whole number,"{""date"":""2020-04-22 00:00:00"",""order_id"":""ORD000592"",""customer_id"":""CS_But_25"",""product_id"":""P10022"",""units_sold"":""-3"",""total_amount"":""124"",""store_id"":""St_Atl_Ge""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000695,ERROR,units_sold must be a positive whole number,"{""date"":""2020-07-06 00:00:00"",""order_id"":""ORD000695"",""customer_id"":""CS_Jar_24"",""product_id"":""P10100"",""units_sold"":""-2"",""total_amount"":""60"",""store_id"":""St_Atl_Ge""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
sales,ORD000845,ERROR,units_sold must be a positive whole number,"{""date"":""2020-08-02 00:00:00"",""order_id"":""ORD000845"",""customer_id"":""CS_Har_42"",""product_id"":""P10023"",""units_sold"":""0"",""total_amount"":""99"",""store_id"":""St_Bal_Ma""}",6d56b1da-784b-4205-8acb-6f2daed766a5,2026-11-09T16:40:43.500Z
